In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio
import cv2
from scipy.ndimage import uniform_filter, sobel

# ==========================================
# 1. Core Model Implementation
# ==========================================
class LogisticRegressionScratch:
    def __init__(self, lr=0.5, n_iters=2000, l2=1e-3):
        self.lr, self.n_iters, self.l2 = lr, n_iters, l2
        self.w, self.b, self.loss_history = None, 0.0, []

    @staticmethod
    def _sigmoid(z):
        out = np.empty_like(z)
        pos = z >= 0
        out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
        exp_z = np.exp(z[~pos])
        out[~pos] = exp_z / (1.0 + exp_z)
        return out

    def _bce_loss(self, y, p):
        p = np.clip(p, 1e-12, 1.0 - 1e-12)
        loss = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
        return loss + (self.l2 / (2*len(y))) * np.sum(self.w**2)

    def fit(self, X, y):
        m, d = X.shape; self.w, self.b = np.zeros(d), 0.0
        for it in range(self.n_iters):
            p = self._sigmoid(X @ self.w + self.b)
            error = p - y
            grad_w = (X.T @ error)/m + (self.l2/m)*self.w
            grad_b = np.mean(error)
            self.w -= self.lr * grad_w; self.b -= self.lr * grad_b
            self.loss_history.append(self._bce_loss(y, p))
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# ==========================================
# 2. Feature Extraction & Preprocessing
# ==========================================
def extract_pixel_features(gray):
    gx = sobel(gray, axis=1); gy = sobel(gray, axis=0)
    grad_mag = np.sqrt(gx**2 + gy**2)
    local_mean = uniform_filter(gray, size=3)
    local_sq = uniform_filter(gray**2, size=3)
    local_std = np.sqrt(np.clip(local_sq - local_mean**2, 0, None))
    return np.stack([gray, grad_mag, gx, gy, local_mean, local_std], axis=-1)

class StandardScaler:
    def fit(self, X):
        self.mean, self.std = X.mean(axis=0), X.std(axis=0) + 1e-8
        return self
    def transform(self, X):
        return (X - self.mean) / self.std

# ==========================================
# 3. Data Loading & Sampling (BSDS500)
# ==========================================
def load_bsds500_data(image_dir, gt_dir, max_images=10):
    '''Loads images, extracts features, and creates ground truth boundary labels'''
    X_all, y_all = [], []
    if not os.path.exists(image_dir) or not os.path.exists(gt_dir):
        print(f"Warning: Dataset directories not found. Please provide valid paths.")
        return np.array([]), np.array([])
        
    img_paths = glob.glob(os.path.join(image_dir, '*.jpg'))[:max_images]
    for img_path in img_paths:
        base = os.path.splitext(os.path.basename(img_path))[0]
        gt_path = os.path.join(gt_dir, base + '.mat')
        if not os.path.exists(gt_path): continue
            
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None: continue
            
        # Extract 6 features per pixel
        features = extract_pixel_features(img)
        
        # Load .mat annotations
        mat = sio.loadmat(gt_path)
        annotators = mat['groundTruth'][0]
        boundaries = np.stack([anno[0][0][1] for anno in annotators], axis=0)
        
        # Consensus: >= 50% agreement
        consensus = (np.mean(boundaries, axis=0) >= 0.5).astype(int)
        
        # Flatten and append
        X_all.append(features.reshape(-1, features.shape[-1]))
        y_all.append(consensus.reshape(-1))
        
    return np.vstack(X_all), np.concatenate(y_all)

def balanced_sampling(X, y, samples_per_class=250):
    '''Samples 250 boundary and 250 non-boundary pixels'''
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    
    if len(pos_idx) > samples_per_class:
        pos_idx = np.random.choice(pos_idx, samples_per_class, replace=False)
    if len(neg_idx) > samples_per_class:
        neg_idx = np.random.choice(neg_idx, samples_per_class, replace=False)
        
    idx = np.concatenate([pos_idx, neg_idx])
    np.random.shuffle(idx)
    return X[idx], y[idx]

# ==========================================
# 4. Evaluation Metrics
# ==========================================
def classification_report(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*prec*rec / (prec + rec) if (prec + rec) else 0.0
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}

# ==========================================
# 5. Main Execution Block
# ==========================================
if __name__ == '__main__':
    # Update these paths to your BSDS500 dataset location
    TRAIN_IMG_DIR = './BSR/BSDS500/data/images/train'
    TRAIN_GT_DIR = './BSR/BSDS500/data/groundTruth/train'
    TEST_IMG_DIR = './BSR/BSDS500/data/images/test'
    TEST_GT_DIR = './BSR/BSDS500/data/groundTruth/test'
    
    print("Loading and sampling training data...")
    X_train_full, y_train_full = load_bsds500_data(TRAIN_IMG_DIR, TRAIN_GT_DIR, max_images=5)
    
    if len(X_train_full) > 0:
        X_train, y_train = balanced_sampling(X_train_full, y_train_full, samples_per_class=250)
        
        print("Loading and sampling testing data...")
        X_test_full, y_test_full = load_bsds500_data(TEST_IMG_DIR, TEST_GT_DIR, max_images=2)
        X_test, y_test = balanced_sampling(X_test_full, y_test_full, samples_per_class=250)
        
        print("Standardizing features...")
        scaler = StandardScaler()
        X_train = scaler.fit(X_train).transform(X_train)
        X_test = scaler.transform(X_test)
        
        print("Training Logistic Regression Model...")
        model = LogisticRegressionScratch(lr=0.5, n_iters=2000, l2=1e-3)
        model.fit(X_train, y_train)
        
        print("Evaluating Model...")
        y_pred = model.predict(X_test)
        metrics = classification_report(y_test, y_pred)
        print("Test Metrics:", metrics)
    else:
        print("Dataset not loaded. Please ensure dataset paths are correct to run the full pipeline.")
